In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    RocCurveDisplay,
)
import matplotlib.pyplot as plt

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42

In [ ]:
from google.colab import drive
drive.mount ('/content/drive')

In [ ]:
dataset_path = ' /content/drive/MyDrive/creditcard.csv'

In [ ]:
df = pd.read_csv(dataset_path.strip())

In [ ]:
def _make_synthetic_fraud_data(n_samples=50_000, fraud_rate=0.0017):
    rng = np.random.RandomState(RANDOM_STATE)
    n_fraud = max(int(n_samples * fraud_rate), 20)
    n_legit = n_samples - n_fraud

    n_features = 28  # mimics V1..V28
    legit = rng.normal(loc=0.0, scale=1.0, size=(n_legit, n_features))
    # Fraud cases pushed into a different region of feature space
    fraud = rng.normal(loc=2.5, scale=1.5, size=(n_fraud, n_features))

    X = np.vstack([legit, fraud])
    y = np.array([0] * n_legit + [1] * n_fraud)

    amount = np.abs(rng.normal(loc=88, scale=250, size=n_samples))
    time = np.sort(rng.uniform(0, 172792, size=n_samples))

    cols = [f"V{i}" for i in range(1, n_features + 1)]
    df = pd.DataFrame(X, columns=cols)
    df.insert(0, "Time", time)
    df["Amount"] = amount
    df["Class"] = y
    return df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

In [ ]:
# 2. SPLIT FIRST — before any SMOTE or scaling touches the data
# ---------------------------------------------------------------------------
def split_data(df, test_size=0.2):
    X = df.drop(columns=["Class"])
    y = df["Class"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=y,  # preserve the real-world 99.83/0.17 split in both sets
    )
    print(f"\nTrain size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")
    print(f"Train fraud rate: {y_train.mean()*100:.4f}%")
    print(f"Test fraud rate:  {y_test.mean()*100:.4f}%  (untouched, real-world ratio)")
    return X_train, X_test, y_train, y_test

In [ ]:
# 3. BUILD PIPELINES (imblearn, not sklearn - so SMOTE lives safely inside CV)
# ---------------------------------------------------------------------------
def build_logreg_pipeline():
    return ImbPipeline(steps=[
        ("scaler", StandardScaler()),          # LR is scale-sensitive
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ])


def build_rf_pipeline():
    return ImbPipeline(steps=[
        ("smote", SMOTE(random_state=RANDOM_STATE)),  # trees don't need scaling
        ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    ])

In [ ]:
# 4. HYPERPARAMETER TUNING - GridSearchCV re-applies SMOTE inside every fold
# ---------------------------------------------------------------------------
def tune_model(pipeline, param_grid, X_train, y_train, scoring="recall"):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    grid = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring=scoring,   # optimize for recall (catching fraud), tune to taste
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )
    grid.fit(X_train, y_train)
    print(f"\nBest params: {grid.best_params_}")
    print(f"Best CV {scoring}: {grid.best_score_:.4f}")
    return grid.best_estimator_

In [ ]:
def plot_roc_curves(results_dict, y_test, save_path="roc_curves.png"):
    fig, ax = plt.subplots(figsize=(7, 6))
    for name, res in results_dict.items():
        RocCurveDisplay.from_predictions(y_test, res["y_proba"], name=name, ax=ax)
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
    ax.set_title("ROC Curves — Fraud Detection Models")
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    print(f"\nROC curve plot saved to {save_path}")

In [ ]:
def main():
    df = load_data("creditcard.csv")
    X_train, X_test, y_train, y_test = split_data(df)

    results = {}

In [ ]:
# Define X_train, X_test, y_train, y_test, and results in the global scope
# as they are needed for the tuning and evaluation steps.
# The 'df' DataFrame is already available from previous cell execution.
X_train, X_test, y_train, y_test = split_data(df)
results = {}

    # ---- Logistic Regression ----
print("\n" + "#" * 60)
print("# Tuning Logistic Regression pipeline")
print("#" * 60)
lr_pipeline = build_logreg_pipeline()
lr_param_grid = {
        "smote__k_neighbors": [3, 5],
        "classifier__C": [0.01, 0.1, 1.0],
    }
best_lr = tune_model(lr_pipeline, lr_param_grid, X_train, y_train)


In [ ]:
results["Logistic_regression"] = evaluate_model(best_lr, X_test, y_test, "Logistic Regression")

In [ ]:
# ---- Random Forest ----
print("\n" + "#" * 60)
print("# Tuning Random Forest pipeline")
print("#" * 60)
rf_pipeline = build_rf_pipeline()
rf_param_grid = {
    "smote__k_neighbors": [5],
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [10, 20, None],
}
best_rf = tune_model(rf_pipeline, rf_param_grid, X_train, y_train)
results["Random Forest"] = evaluate_model(best_rf, X_test, y_test, "Random Forest")

In [ ]:
   # ---- Compare ----
    print("\n" + "=" * 60)
    print("MODEL COMPARISON (test set, never touched by SMOTE)")
    print("=" * 60)
    comp = pd.DataFrame({k: {m: v[m] for m in ["precision", "recall", "f1", "roc_auc"]}
                          for k, v in results.items()}).T
    print(comp.round(4))

    plot_roc_curves(results, y_test)


if __name__ == "__main__":
    main()